In [ ]:
import h5py
import numpy as np
import json
import os
from datetime import datetime

# Definisi elemen target
# versi 35
BASE_ELEMENTS = ["Si", "Al", "Fe", "Ca", "O", "Na", "N", "Ni", "Cr", "Cl", "Mg", "C", "S", "Ar", "Ti", "Mn", "Co"]
REQUIRED_ELEMENTS = [f"{elem}_{i}" for elem in BASE_ELEMENTS for i in [1, 2]]

#versi 9
# BASE_ELEMENTS = ["Al", "Fe", "Ca", "Mg"]
# REQUIRED_ELEMENTS = [f"{elem}_{i}" for elem in BASE_ELEMENTS for i in [1, 2]]

# Konfigurasi
CONFIG = {
    "data": {
        "dataset_path": "/Volumes/Data/birrulwldain/20/spectral_dataset_final_20k.h5",
        "element_map_path": "element-map-35.json",
        "train_split": "train",
        "val_split": "validation",
        "max_train_samples": 20000,
        "max_val_samples": 20000,
    },
    "results_dir": "r"
}

def load_element_map(element_map_path):
    """Memuat mapping elemen dari file JSON sebagai dictionary."""
    with open(element_map_path, 'r') as f:
        element_map = json.load(f)
    return element_map

def check_dataset(dataset_path, element_map_path, train_split, val_split, max_train_samples, max_val_samples):
    """Memeriksa proporsi atom dalam dataset."""

    # Muat element map sebagai dictionary
    element_map = load_element_map(element_map_path)
    print(f"{datetime.now()} - Element map dimuat: {list(element_map.keys())}")
    print(f"{datetime.now()} - Number of elements/ions in element_map: {len(element_map)}")

    # Verifikasi panjang nilai one-hot
    if not all(len(one_hot) == 35 for one_hot in element_map.values()):
        print(f"{datetime.now()} - Peringatan: Tidak semua nilai one-hot memiliki panjang 35.")
        return

    # Mapping elemen dasar ke indeks kelas berdasarkan one-hot
    element_to_class = {}
    for elem in BASE_ELEMENTS:
        element_to_class[elem] = []
        for class_name, one_hot in element_map.items():
            if class_name.startswith(elem) and class_name in REQUIRED_ELEMENTS:
                idx = np.argmax(one_hot)  # Indeks kelas berdasarkan one-hot
                element_to_class[elem].append(idx)

    # Mapping kelas ke elemen dasar berdasarkan one-hot
    class_to_element = {}
    for class_name, one_hot in element_map.items():
        idx = np.argmax(one_hot)
        elem = next((e for e in BASE_ELEMENTS if class_name.startswith(e)), "background")
        class_to_element[idx] = elem

    # Inisialisasi dictionary untuk menyimpan distribusi
    split_distributions = {"train": {}, "validation": {}}

# Hapus bagian element_counts dan element_proportions
    with h5py.File(dataset_path, 'r') as h5_file:
        for split, max_samples in [(train_split, max_train_samples), (val_split, max_val_samples)]:
            if split not in h5_file:
                print(f"{datetime.now()} - Split '{split}' tidak ditemukan dalam {dataset_path}")
                continue

            group = h5_file[split]
            spectra = group['spectra'][:max_samples]
            labels = group['labels'][:max_samples]
            print(f"{datetime.now()} - Memproses split: {split}, Spectra shape: {spectra.shape}, Labels shape: {labels.shape}")

            # Hitung distribusi kelas di semua posisi
            all_labels = labels.reshape(-1, 35)  # Flattening ke semua posisi
            class_counts = np.sum(all_labels, axis=0)  # Jumlah 1 di setiap kelas
            total_positions = all_labels.shape[0]  # Total posisi (sampel × 4096)

            # Distribusi per kelas
            split_distributions[split] = {}
            for idx, count in enumerate(class_counts):
                class_name = next((name for name, one_hot in element_map.items() if np.argmax(one_hot) == idx), f"Class_{idx}")
                proportion = count / total_positions * 100
                split_distributions[split][class_name] = {"count": int(count), "proportion (%)": round(proportion, 2)}

            # Cetak hasil
            print(f"\n{datetime.now()} - Distribusi Kelas untuk {split} (Total Posisi: {total_positions:,}):")
            for class_name, stats in split_distributions[split].items():
                print(f"  {class_name}: {stats['count']:,} positions ({stats['proportion (%)']:.2f}%)")

            # Simpan hasil ke file
            os.makedirs(CONFIG["results_dir"], exist_ok=True)
            report_path = os.path.join(CONFIG["results_dir"], f"dataset_distribution_{split}.txt")
            with open(report_path, "w") as f:
                f.write(f"Dataset Distribution Report - {split} (Generated: {datetime.now()})\n")
                f.write(f"Total Positions: {total_positions:,}\n\n")
                f.write("Class Distribution:\n")
                for class_name, stats in split_distributions[split].items():
                    f.write(f"  {class_name}: {stats['count']:,} positions ({stats['proportion (%)']:.2f}%)\n")
            print(f"{datetime.now()} - Laporan distribusi disimpan ke {report_path}")

if __name__ == "__main__":
    check_dataset(
        CONFIG["data"]["dataset_path"],
        CONFIG["data"]["element_map_path"],
        CONFIG["data"]["train_split"],
        CONFIG["data"]["val_split"],
        CONFIG["data"]["max_train_samples"],
        CONFIG["data"]["max_val_samples"]
    )

2025-06-14 17:19:57.886299 - Element map dimuat: ['background', 'Si_1', 'Si_2', 'Al_1', 'Al_2', 'Fe_1', 'Fe_2', 'Ca_1', 'Ca_2', 'O_1', 'O_2', 'Na_1', 'Na_2', 'N_1', 'N_2', 'Ni_1', 'Ni_2', 'Cr_1', 'Cr_2', 'Cl_1', 'Cl_2', 'Mg_1', 'Mg_2', 'C_1', 'C_2', 'S_1', 'S_2', 'Ar_1', 'Ar_2', 'Ti_1', 'Ti_2', 'Mn_1', 'Mn_2', 'Co_1', 'Co_2']
2025-06-14 17:19:57.886830 - Number of elements/ions in element_map: 35


In [31]:
import h5py
import numpy as np
import json
import os
import matplotlib.pyplot as plt
import random
import matplotlib.patches as mpatches
from collections import defaultdict
import matplotlib as mpl

# Set Matplotlib untuk tampilan ilmiah
mpl.rcParams['font.family'] = 'Times New Roman'
mpl.rcParams['font.size'] = 10
mpl.rcParams['axes.titlesize'] = 14
mpl.rcParams['axes.labelsize'] = 12
mpl.rcParams['xtick.labelsize'] = 10
mpl.rcParams['ytick.labelsize'] = 10
mpl.rcParams['legend.fontsize'] = 9

def generate_plots_and_latex(h5_path, map_path, num_plots=5, output_dir="/Volumes/Data/birrulwldain/spectral_report_v12"):
    """
    Membuka file HDF5, menghasilkan plot spektrum sebagai gambar PNG terpisah,
    dan membuat dokumen LaTeX dengan setiap plot dalam satu halaman penuh horizontal.

    Parameters:
    - h5_path: Path ke file HDF5 dataset
    - map_path: Path ke file JSON element map
    - num_plots: Jumlah plot acak yang akan dibuat
    - output_dir: Direktori untuk menyimpan file output
    """
    # Validasi file input
    if not os.path.exists(h5_path):
        print(f"Error: File dataset tidak ditemukan di {h5_path}")
        return
    if not os.path.exists(map_path):
        print(f"Error: File element map tidak ditemukan di {map_path}")
        return

    # Buat direktori output
    os.makedirs(output_dir, exist_ok=True)
    
    # Muat data pendukung
    with open(map_path, 'r') as f:
        element_map = json.load(f)
    
    idx_to_element = {np.argmax(v): k for k, v in element_map.items()}
    element_to_idx = {k: np.argmax(v) for k, v in element_map.items()}

    # Inisialisasi data untuk LaTeX
    latex_data = []

    with h5py.File(h5_path, 'r') as f:
        if 'train' not in f:
            print("Error: Grup 'train' tidak ditemukan dalam file HDF5.")
            return
            
        train_group = f['train']
        num_train_samples = train_group['spectra'].shape[0]
        
        if num_train_samples == 0:
            print("Peringatan: Tidak ada sampel dalam grup 'train'.")
            return

        if num_train_samples < num_plots:
            print(f"Peringatan: Jumlah sampel ({num_train_samples}) lebih sedikit dari jumlah plot yang diminta ({num_plots}).")
            num_plots = num_train_samples

        random_indices = random.sample(range(num_train_samples), num_plots)
        wavelengths = f['wavelengths'][:]
        
        colors_cmap = plt.colormaps['tab20']
        background_idx = element_to_idx.get('background', -1)
        
        for sample_idx in random_indices:
            spectrum = train_group['spectra'][sample_idx]
            labels = train_group['labels'][sample_idx]
            
            # Dapatkan metadata
            metadata = {}
            sample_id = f"Sample_{sample_idx:04d}"
            temp, n_e = 'N/A', 'N/A'
            initial_composition_data = {}
            try:
                metadata_str = train_group['atom_percentages'][sample_idx].decode('utf-8')
                metadata = json.loads(metadata_str)
                sample_id = metadata.get('sample_id', f"Sample_{sample_idx:04d}")
                temp = metadata.get('temperature', 'N/A')
                n_e = metadata.get('electron_density', 'N/A')
                
                for ion_key, perc in metadata.items():
                    if ion_key not in ['temperature', 'electron_density', 'delta_E_max', 'sample_id']:
                        initial_composition_data[ion_key] = perc

            except (KeyError, IndexError, json.JSONDecodeError) as e:
                print(f"Peringatan: Gagal memuat metadata untuk ID {sample_idx}: {e}")

            # Hitung posisi aktual (jumlah piksel)
            actual_positions = defaultdict(int)
            total_pixels = labels.shape[0]
            for label_idx in range(labels.shape[1]):
                element_name = idx_to_element.get(label_idx)
                if element_name:
                    active_pixels = np.sum(labels[:, label_idx])
                    actual_positions[element_name] = active_pixels
            
            total_sum_positions = sum(actual_positions.values())

            # Hitung rasio sinyal-ke-noise sederhana (saran informasi tambahan)
            noise_level = np.std(spectrum)
            signal_level = np.max(spectrum)
            snr = signal_level / noise_level if noise_level > 0 else 0

            # --- Generate Plot ---
            fig, ax = plt.subplots(figsize=(10.0, 6.0), dpi=300)  # Ukuran lebih besar untuk penuh halaman
            
            ax.plot(wavelengths, spectrum, color='black', linewidth=1.2, label='Spektrum Gabungan')
            legend_handles = [mpatches.Patch(color='black', label='Spektrum Gabungan')]
            
            for element_idx in range(labels.shape[1]):
                if element_idx == background_idx:
                    continue

                element_name = idx_to_element.get(element_idx)
                if not element_name:
                    continue

                pixels_with_this_element = np.where(labels[:, element_idx] == 1)[0]
                
                if len(pixels_with_this_element) > 0:
                    spectrum_portion = np.zeros_like(spectrum)
                    spectrum_portion[pixels_with_this_element] = spectrum[pixels_with_this_element]

                    color = colors_cmap(element_idx / len(element_map))
                    ax.fill_between(wavelengths, 0, spectrum_portion, 
                                    facecolor=color, alpha=0.3, label=f'{element_name}')
                    ax.plot(wavelengths, spectrum_portion, color=color, linewidth=0.6)
                    
                    if len(pixels_with_this_element) > 0:
                        max_intensity_idx = pixels_with_this_element[np.argmax(spectrum[pixels_with_this_element])]
                        if spectrum[max_intensity_idx] > 0.05:
                            ax.annotate(
                                element_name,
                                (wavelengths[max_intensity_idx], spectrum[max_intensity_idx]),
                                textcoords="offset points", xytext=(0, 10), ha='center',
                                fontsize=9, color=color,
                                arrowprops=dict(facecolor=color, shrink=0.05, width=0.5, headwidth=4, alpha=0.7)
                            )
                    
                    legend_handles.append(mpatches.Patch(color=color, label=element_name))

            # Background
            background_pixels = np.where(labels[:, background_idx] == 1)[0]
            if len(background_pixels) > 0:
                background_portion = np.zeros_like(spectrum)
                background_portion[background_pixels] = spectrum[background_pixels]
                ax.fill_between(wavelengths, 0, background_portion, facecolor='lightgray', alpha=0.2, label='Latar Belakang')
                legend_handles.append(mpatches.Patch(color='lightgray', label='Latar Belakang'))

            # Kustomisasi plot dengan legenda di sudut atas kanan, transparan
            ax.set_title(f"Simulasi Spektrum (ID: {sample_id})", fontsize=14, pad=15, weight='bold')
            ax.set_xlabel("Panjang Gelombang (nm)", fontsize=12, labelpad=10)
            ax.set_ylabel("Intensitas Ternormalisasi", fontsize=12, labelpad=10)
            ax.grid(True, which='both', linestyle='--', linewidth=0.3, alpha=0.7)
            ax.set_xlim(wavelengths.min(), wavelengths.max())
            ax.set_ylim(0, max(1.0, np.max(spectrum) * 1.1))
            
            # Legenda di sudut atas kanan dengan transparansi
            unique_labels_dict = {handle.get_label(): handle for handle in legend_handles}
            ax.legend(handles=list(unique_labels_dict.values()), loc='upper right', fontsize=7, frameon=True, 
                      edgecolor='black', facecolor='white', framealpha=0.7)

            # Simpan plot sebagai PNG terpisah
            plot_filename = f"spectrum_{sample_id}.png"
            plot_filepath = os.path.join(output_dir, plot_filename)
            plt.tight_layout()
            plt.savefig(plot_filepath, format='png', dpi=300, bbox_inches='tight')
            plt.close(fig)
            print(f"Plot disimpan: {plot_filepath}")

            # Kumpulkan data untuk LaTeX
            table_rows = []
            sorted_elements = sorted(list(set(k.split('_')[0] for k in initial_composition_data.keys()) | set(actual_positions.keys())))
            if 'background' in sorted_elements:
                sorted_elements.remove('background')
                sorted_elements.append('background')

            for elem_name in sorted_elements:
                initial_comp_val = "N/A"
                if f"{elem_name}_1" in initial_composition_data or f"{elem_name}_2" in initial_composition_data:
                    ion1_perc = initial_composition_data.get(f"{elem_name}_1", 0)
                    ion2_perc = initial_composition_data.get(f"{elem_name}_2", 0)
                    initial_comp_val = f"{ion1_perc + ion2_perc:.2f}"
                elif elem_name == 'background':
                    initial_comp_val = 'N/A'
                else:
                    initial_comp_val = "0.00"

                actual_pos_val = actual_positions.get(elem_name, 0)
                table_rows.append({
                    'element': elem_name,
                    'initial_comp': initial_comp_val,
                    'actual_pos': actual_pos_val
                })

            latex_data.append({
                'sample_id': sample_id,
                'temperature': f"{float(temp):.0f}" if temp != 'N/A' else 'N/A',
                'electron_density': f"{float(n_e):.2e}" if n_e != 'N/A' else 'N/A',
                'total_pixels': total_pixels,
                'snr': f"{snr:.2f}",
                'total_actual': total_sum_positions,
                'table_rows': table_rows,
                'plot_filename': plot_filename
            })

    # --- Generate LaTeX Document ---
    latex_content = r"""
\documentclass[a4paper,11pt]{article}
\usepackage{geometry}
\geometry{margin=1cm} % Margin lebih kecil untuk memaksimalkan ruang plot
\usepackage{booktabs}
\usepackage{graphicx}
\usepackage{amsmath}
\usepackage{siunitx}
\usepackage{caption}
\usepackage{times}
\usepackage[utf8]{inputenc}
\usepackage{tocloft}
\usepackage{float}

\title{Spectral Simulation Report}
\author{Automated Analysis}
\date{\today}

\begin{document}

\maketitle
\tableofcontents
\clearpage



"""

    for sample in latex_data:
        sample_id = sample['sample_id']
        temp = sample['temperature']
        n_e = sample['electron_density']
        total_pixels = sample['total_pixels']
        snr = sample['snr']
        total_actual = sample['total_actual']
        table_rows = sample['table_rows']
        plot_filename = sample['plot_filename']

        latex_content += f"""
\\section{{Sample: {sample_id}}}
\\begin{{minipage}}[t]{{0.48\\textwidth}}
    \\subsection{{Simulation Parameters}}
    \\centering
    \\begin{{table}}[H]
        \\centering
        \\small
        \\begin{{tabular}}{{ll}}
            \\toprule
            \\textbf{{Parameter}} & \\textbf{{Value}} \\\\
            \\midrule
            Temperature ($T$) & \\SI{{{temp}}}{{\\kelvin}} \\\\
            Electron Density ($n_e$) & \\SI{{{n_e}}}{{\\per\\cubic\\centi\\meter}} \\\\
            Total Pixels & \\num{{{total_pixels}}} \\\\
            Signal-to-Noise Ratio (SNR) & \\num{{{snr}}} \\\\
            \\bottomrule
        \\end{{tabular}}
        \\caption{{Simulation parameters and additional metrics for sample {sample_id}.}}
        \\label{{tab:params_{sample_id}}}
    \\end{{table}}
\\end{{minipage}}\\hfill
\\begin{{minipage}}[t]{{0.48\\textwidth}}
    \\subsection{{Composition Analysis}}
    \\centering
    \\begin{{table}}[H]
        \\centering
        \\small
        \\begin{{tabular}}{{lcc}}
            \\toprule
            \\textbf{{Element}} & \\textbf{{Initial Composition (\\%)}} & \\textbf{{Actual Positions}} \\\\
            \\midrule
"""

        for row in table_rows:
            element = row['element'].replace('_', '\\_')
            initial_comp = row['initial_comp']
            actual_pos = row['actual_pos']
            latex_content += f"            {element} & {initial_comp} & {actual_pos} \\\\\n"

        latex_content += f"""
            \\midrule
            \\textbf{{Total Positions}} & & {total_actual} \\\\
            \\bottomrule
        \\end{{tabular}}
        \\caption{{Composition analysis for sample {sample_id}.}}
        \\label{{tab:comp_{sample_id}}}
    \\end{{table}}
\\end{{minipage}}
\\subsection{{Spectral Plot}}
\\begin{{figure}}[h]
    \\centering
    \\includegraphics[width=\\textwidth,angle=0]{{/Volumes/Data/birrulwldain/spectral_report_v12/{plot_filename}}}
    \\caption{{Spectral simulation plot for sample {sample_id} with multi-label encoding and detailed composition analysis.}}
    \\label{{fig:spectrum_{sample_id}}}
    \\addcontentsline{{toc}}{{subsection}}{{Spectral Plot for Sample {sample_id}}}
\\end{{figure}}
\\clearpage
"""

    latex_content += r"""
\end{document}
"""

    # Simpan file LaTeX
    latex_filepath = os.path.join(output_dir, "spectral_report.tex")
    with open(latex_filepath, 'w', encoding='utf-8') as f:
        f.write(latex_content)
    print(f"Dokumen LaTeX disimpan: {latex_filepath}")

    print(f"\nInstruksi: Untuk menghasilkan PDF, jalankan perintah berikut di direktori {output_dir}:\n"
          f"pdflatex {latex_filepath}\n"
          "Pastikan pdflatex terinstal dan semua file gambar (PNG) ada di direktori yang sama. Jalankan dua kali untuk memperbarui referensi.")

# --- Konfigurasi Path ---
H5_FILE_PATH = "dataset-20.h5"  # Ganti dengan path dataset Anda
MAP_FILE_PATH = "element-map-18a.json"  # Ganti dengan path element map Anda
OUTPUT_PLOTS_DIR = "spectral_report_v12"  # Direktori output

# Jalankan fungsi
generate_plots_and_latex(H5_FILE_PATH, MAP_FILE_PATH, num_plots=5, output_dir=OUTPUT_PLOTS_DIR)

Plot disimpan: spectral_report_v12/spectrum_15167.png
Plot disimpan: spectral_report_v12/spectrum_7238.png
Plot disimpan: spectral_report_v12/spectrum_15941.png
Plot disimpan: spectral_report_v12/spectrum_19431.png
Plot disimpan: spectral_report_v12/spectrum_15612.png
Dokumen LaTeX disimpan: spectral_report_v12/spectral_report.tex

Instruksi: Untuk menghasilkan PDF, jalankan perintah berikut di direktori spectral_report_v12:
pdflatex spectral_report_v12/spectral_report.tex
Pastikan pdflatex terinstal dan semua file gambar (PNG) ada di direktori yang sama. Jalankan dua kali untuk memperbarui referensi.


In [28]:
import numpy as np
import pandas as pd
import torch 
import h5py
import json
import os
import re
import logging
from scipy.signal.windows import gaussian
from typing import List, Dict, Tuple, Optional
import matplotlib.pyplot as plt

# =============================================================================
# BAGIAN 1: SALIN KELAS-KELAS INTI DARI SKRIP ANDA
# (Tidak perlu mengubah bagian ini, ini adalah library yang Anda buat)
# =============================================================================

# Konfigurasi disederhanakan untuk alat ini
SIMULATION_CONFIG = {
    "resolution": 4096,
    "wl_range": (200, 900),
    "sigma": 0.1,
    "target_max_intensity": 0.8,
    "convolution_sigma": 0.1,
}

# Setup logger sederhana untuk output konsol
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger('InteractivePlotter')

class DataFetcher:
    """Mengambil data spektral dari file HDF5 NIST untuk elemen dan ion tertentu."""
    def __init__(self, hdf_path: str):
        self.hdf_path = hdf_path
        if not os.path.exists(hdf_path):
            raise FileNotFoundError(f"File data NIST tidak ditemukan di: {hdf_path}")
        self.logger = logging.getLogger('InteractivePlotter')

    def get_nist_data(self, element: str, sp_num: int) -> List[List]:
        elem_key = f"{element}_{sp_num}"
        try:
            with pd.HDFStore(self.hdf_path, mode='r') as store:
                df = store.get('nist_spectroscopy_data')
                filtered_df = df[(df['element'] == element) & (df['sp_num'] == sp_num)]
                required_columns = ['ritz_wl_air(nm)', 'Aki(s^-1)', 'Ek(eV)', 'Ei(eV)', 'g_i', 'g_k']

                if filtered_df.empty: return []
                filtered_df = filtered_df.dropna(subset=required_columns).copy()
                
                filtered_df['ritz_wl_air(nm)'] = pd.to_numeric(filtered_df['ritz_wl_air(nm)'], errors='coerce')
                for col in ['Ek(eV)', 'Ei(eV)', 'Aki(s^-1)', 'g_i', 'g_k']:
                     filtered_df.loc[:, col] = pd.to_numeric(
                        filtered_df[col].apply(lambda x: re.sub(r'[^\d.-]', '', str(x)) if isinstance(x, str) else x),
                        errors='coerce'
                    )

                filtered_df = filtered_df.dropna(subset=required_columns)
                return filtered_df[required_columns].values.tolist()
        except Exception as e:
            self.logger.error(f"Error mengambil data NIST untuk {elem_key}: {str(e)}")
            return []

class SpectrumSimulator:
    """Mensimulasikan spektrum emisi untuk satu elemen dan ion."""
    def __init__(self, nist_data: List[List], element: str, ion: int, config: Dict):
        self.nist_data = nist_data
        self.element = element
        self.ion = ion
        self.config = config
        self.resolution = config["resolution"]
        self.wl_range = config["wl_range"]
        self.sigma = config["sigma"]
        self.wavelengths = np.linspace(self.wl_range[0], self.wl_range[1], self.resolution, dtype=np.float32)
        self.logger = logging.getLogger('InteractivePlotter')

    def _partition_function(self, energy_levels: List[float], degeneracies: List[float], temperature: float) -> float:
        k_B = 8.617333262145e-5
        return sum(g * np.exp(-E / (k_B * temperature)) for g, E in zip(degeneracies, energy_levels) if E is not None) or 1.0

    def _calculate_intensity(self, temperature: float, energy: float, degeneracy: float, einstein_coeff: float, Z: float) -> float:
        k_B = 8.617333262145e-5
        return (degeneracy * einstein_coeff * np.exp(-energy / (k_B * temperature))) / Z

    def simulate_single_temp(self, temp: float, atom_percentage: float = 1.0) -> Optional[np.ndarray]:
        if not self.nist_data: return None
        levels = {}
        for data in self.nist_data:
            _, _, Ek, Ei, gi, gk = data
            if all(v is not None for v in [Ek, Ei, gi, gk]):
                levels[float(Ei)] = float(gi)
                levels[float(Ek)] = float(gk)

        if not levels: return None
        energy_levels, degeneracies = list(levels.keys()), list(levels.values())
        Z = self._partition_function(energy_levels, degeneracies, temp)
        
        spectrum = np.zeros(self.resolution, dtype=np.float32)
        wavelength_step = (self.wl_range[1] - self.wl_range[0]) / (self.resolution - 1)
        sigma_points = self.sigma / wavelength_step

        for data in self.nist_data:
            wl, Aki, Ek, _, _, gk = data
            if any(v is None for v in [wl, Aki, Ek, gk]): continue
            
            intensity = self._calculate_intensity(temp, float(Ek), float(gk), float(Aki), Z)
            idx = np.searchsorted(self.wavelengths, float(wl))

            if 0 <= idx < self.resolution:
                kernel_size = int(6 * sigma_points) | 1
                kernel_half_width = kernel_size // 2
                kernel_x = np.arange(-kernel_half_width, kernel_half_width + 1)
                kernel = np.exp(-0.5 * (kernel_x / sigma_points)**2)
                
                target_start, target_end = idx - kernel_half_width, idx + kernel_half_width + 1
                valid_target_start, valid_target_end = max(0, target_start), min(self.resolution, target_end)
                valid_kernel_start = valid_target_start - target_start
                valid_kernel_end = valid_kernel_start + (valid_target_end - valid_target_start)

                if valid_target_start < valid_target_end:
                    final_kernel_slice = kernel[valid_kernel_start:valid_kernel_end]
                    spectrum[valid_target_start:valid_target_end] += intensity * atom_percentage * final_kernel_slice
        return spectrum

# =============================================================================
# BAGIAN 2: KODE INTERAKTIF UNTUK SIMULASI DAN PLOT
# =============================================================================

class InteractiveSimulator:
    def __init__(self, data_dir="/home/bwalidain/birrulwldain/data"):
        self.data_dir = data_dir
        self.nist_path = os.path.join(data_dir, "nist_data(1).h5")
        self.atomic_data_path = os.path.join(data_dir, "atomic_data1.h5")
        self.fetcher = DataFetcher(self.nist_path)
        self.ionization_energies = self._load_ionization_energies()
        self.simulators_cache = {}

    def _load_ionization_energies(self) -> Dict[str, float]:
        """Memuat energi ionisasi dari file HDF5 data atom."""
        if not os.path.exists(self.atomic_data_path):
            logger.warning(f"File data atom tidak ditemukan di {self.atomic_data_path}. Energi ionisasi tidak akan digunakan.")
            return {}
        energies = {}
        try:
            with h5py.File(self.atomic_data_path, 'r') as f:
                df_ionization = pd.DataFrame(f['elements'][:], columns=f['elements'].attrs['columns'])
                # Decode byte strings
                for col in df_ionization.columns:
                    if df_ionization[col].dtype == object:
                       df_ionization[col] = df_ionization[col].str.decode('utf-8')

                for _, row in df_ionization.iterrows():
                    energies[row['Sp. Name']] = float(row['Ionization Energy (eV)'])
        except Exception as e:
            logger.error(f"Gagal memuat energi ionisasi: {e}")
        return energies

    def _saha_ratio(self, ion_energy: float, temp: float, electron_density: float) -> float:
        """Menghitung rasio populasi ion menggunakan Persamaan Saha."""
        if ion_energy == 0: return 0.0
        k_B_eV_K = 8.617333262e-5
        m_e_kg = 9.1093837e-31
        h_eV_s = 4.135667696e-15
        kT_eV = k_B_eV_K * temp
        saha_factor = 2 * (2 * np.pi * m_e_kg * kT_eV * 1.60218e-19 / (h_eV_s * 1.60218e-19)**2)**1.5
        saha_factor /= (electron_density * 1e6)
        return saha_factor * np.exp(-ion_energy / kT_eV)
        
    def get_simulator(self, element: str, ion: int) -> SpectrumSimulator:
        """Membuat atau mengambil simulator dari cache."""
        key = f"{element}_{ion}"
        if key not in self.simulators_cache:
            nist_data = self.fetcher.get_nist_data(element, ion)
            self.simulators_cache[key] = SpectrumSimulator(nist_data, element, ion, SIMULATION_CONFIG)
        return self.simulators_cache[key]

    def run_simulation(self, temp: float, ne: float, composition: Dict[str, float]):
        """Menjalankan simulasi penuh berdasarkan input pengguna."""
        logger.info(f"Memulai simulasi untuk T={temp} K, ne={ne:.2e} cm⁻³")
        
        final_spectrum = np.zeros(SIMULATION_CONFIG['resolution'], dtype=np.float32)
        element_contributions = {}

        # Hitung persentase untuk setiap spesies ion berdasarkan Persamaan Saha
        for element, total_percentage in composition.items():
            ion_energy = self.ionization_energies.get(f"{element} I", 0.0)
            saha_ratio = self._saha_ratio(ion_energy, temp, ne)
            
            frac_neutral = 1 / (1 + saha_ratio)
            frac_ion = 1 - frac_neutral
            
            # Persentase untuk atom netral (ion 1) dan terionisasi (ion 2)
            perc_neutral = total_percentage * frac_neutral
            perc_ion = total_percentage * frac_ion

            # Jalankan simulasi untuk ion 1 (netral)
            sim_neutral = self.get_simulator(element, 1)
            spec_neutral = sim_neutral.simulate_single_temp(temp, perc_neutral / 100.0)
            if spec_neutral is not None:
                element_contributions[f"{element}_1"] = spec_neutral
                final_spectrum += spec_neutral
            
            # Jalankan simulasi untuk ion 2 (terionisasi)
            sim_ion = self.get_simulator(element, 2)
            spec_ion = sim_ion.simulate_single_temp(temp, perc_ion / 100.0)
            if spec_ion is not None:
                element_contributions[f"{element}_2"] = spec_ion
                final_spectrum += spec_ion

        # Normalisasi spektrum akhir
        max_intensity = np.max(final_spectrum)
        if max_intensity > 0:
            final_spectrum /= max_intensity
            final_spectrum *= SIMULATION_CONFIG['target_max_intensity']
            for key in element_contributions:
                element_contributions[key] /= max_intensity
                element_contributions[key] *= SIMULATION_CONFIG['target_max_intensity']

        return final_spectrum, element_contributions

    def plot_results(self, final_spectrum: np.ndarray, contributions: Dict[str, np.ndarray], params: Dict):
        """Membuat plot ilmiah dari hasil simulasi."""
        fig, ax = plt.subplots(figsize=(16, 8))
        
        wavelengths = np.linspace(SIMULATION_CONFIG['wl_range'][0], SIMULATION_CONFIG['wl_range'][1], SIMULATION_CONFIG['resolution'])
        
        # Plot kontribusi setiap elemen/ion
        colors = plt.cm.get_cmap('tab20', len(contributions))
        for i, (name, spec) in enumerate(contributions.items()):
            ax.plot(wavelengths, spec, label=name, color=colors(i), linewidth=0.7)

        # Plot spektrum gabungan di atasnya
        ax.plot(wavelengths, final_spectrum, color='black', linewidth=1.2, label='Spektrum Gabungan')
        
        title = f"Simulasi Spektrum: T={params['temp']} K, ne={params['ne']:.2e} cm⁻³\nKomposisi: {params['comp_str']}"
        ax.set_title(title, fontsize=16)
        ax.set_xlabel("Panjang Gelombang (nm)", fontsize=12)
        ax.set_ylabel("Intensitas Relatif", fontsize=12)
        ax.grid(True, linestyle=':', linewidth=0.5)
        ax.set_xlim(SIMULATION_CONFIG['wl_range'])
        ax.set_ylim(bottom=0)
        ax.legend(loc='upper right')
        
        plt.tight_layout()
        plt.show()

    def start_interactive_session(self):
        """Memulai loop interaktif untuk menerima input dari pengguna."""
        print("\n--- Plotter Spektrum Interaktif ---")
        print("Masukkan 'exit' atau 'quit' kapan saja untuk keluar.")
        
        while True:
            try:
                # 1. Dapatkan input suhu
                temp_str = input("\nMasukkan Suhu [Kelvin] (contoh: 8000): ")
                if temp_str.lower() in ['exit', 'quit']: break
                temp = float(temp_str)

                # 2. Dapatkan input kepadatan elektron
                ne_str = input("Masukkan Kepadatan Elektron [cm⁻³] (contoh: 1e16): ")
                if ne_str.lower() in ['exit', 'quit']: break
                ne = float(ne_str)

                # 3. Dapatkan input komposisi
                comp_str = input("Masukkan Komposisi (Elemen:Persen, ...) (contoh: Fe:80, Si:20): ")
                if comp_str.lower() in ['exit', 'quit']: break
                
                composition = {}
                parts = comp_str.replace(" ", "").split(',')
                for part in parts:
                    elem, perc = part.split(':')
                    composition[elem.strip().capitalize()] = float(perc)
                
                total_perc = sum(composition.values())
                if not np.isclose(total_perc, 100):
                    logger.warning(f"Total persentase adalah {total_perc:.1f}%, bukan 100%. Hasil akan diskalakan.")
                    # Normalisasi persentase
                    for elem in composition:
                        composition[elem] = (composition[elem] / total_perc) * 100

                # Jalankan simulasi dan plot
                spectrum, contributions = self.run_simulation(temp, ne, composition)
                
                plot_params = {'temp': temp, 'ne': ne, 'comp_str': comp_str}
                self.plot_results(spectrum, contributions, plot_params)

            except (ValueError, IndexError) as e:
                logger.error(f"Input tidak valid. Pastikan format benar. Error: {e}")
                continue
            except KeyboardInterrupt:
                print("\nSesi dihentikan oleh pengguna.")
                break
            except Exception as e:
                logger.error(f"Terjadi kesalahan tak terduga: {e}", exc_info=True)
                break
        
        print("\nTerima kasih telah menggunakan plotter interaktif!")


if __name__ == '__main__':
    # Pastikan path ke direktori data Anda sudah benar
    DATA_DIRECTORY = "/home/bwalidain/birrulwldain/data"
    
    # Jalankan sesi interaktif
    interactive_sim = InteractiveSimulator(data_dir=DATA_DIRECTORY)
    interactive_sim.start_interactive_session()

ModuleNotFoundError: No module named 'torch'

In [18]:
import h5py
import numpy as np
import json
import os
from datetime import datetime
from sklearn.model_selection import train_test_split
from collections import Counter
from tqdm import tqdm
from collections import defaultdict
import math

# --- KONFIGURASI ---
# Sesuaikan dengan struktur direktori Anda
CONFIG = {
    "data": {
        "original_dataset_path": "/Volumes/Data/birrulwldain/spectral_dataset-35.h5", # File sumber
        "stratified_dataset_path": "/Volumes/Data/birrulwldain/spectral_dataset-35_s.h5", # File tujuan
        "element_map_path": "/Volumes/Data/birrulwldain/element-map-35.json"
    },
    "model": {
        "seq_length": 4096
    },
    "split": {
        "test_size": 0.3,  # 30% untuk (validasi + test)
        "validation_size": 0.5 # 50% dari 30% -> 15% dari total untuk validasi
    }
}

# --- FUNGSI UTILITAS (dari kode Anda) ---
def get_timestamp():
    """Mendapatkan timestamp saat ini dalam format string."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# --- FUNGSI PEMBUAT DATASET (dari solusi saya, diintegrasikan) ---
# KODE FINAL - Ganti seluruh fungsi create_stratified_dataset dengan ini.

def create_stratified_dataset(original_path, new_path, element_map_path):
    """
    Membaca dataset asli, dan melakukan stratified split secara manual dan tangguh
    untuk menangani kelas yang sangat langka.
    """
    print(f"[{get_timestamp()}] Memulai pembuatan dataset stratified dengan metode manual...")
    
    if not os.path.exists(original_path):
        raise FileNotFoundError(f"File dataset asli tidak ditemukan di: {original_path}")

    # 1. Muat data (Sama)
    print(f"[{get_timestamp()}] Memuat data...")
    with h5py.File(original_path, 'r') as f:
        wavelengths = f['wavelengths'][:]
        all_spectra = np.concatenate([f[g]['spectra'][:] for g in ['train', 'validation', 'test']], axis=0)
        all_labels = np.concatenate([f[g]['labels'][:] for g in ['train', 'validation', 'test']], axis=0)
        all_atom_percentages_str = np.concatenate([f[g]['atom_percentages'][:] for g in ['train', 'validation', 'test']], axis=0)

    # 2. Buat Kunci Stratifikasi (Sama)
    print(f"[{get_timestamp()}] Membuat kunci stratifikasi...")
    strata_keys = []
    for json_str in tqdm(all_atom_percentages_str, desc="Parsing Metadata"):
        data = json.loads(json_str.decode('utf-8'))
        element_percentages = {k: v for k, v in data.items() if k not in ['temperature', 'electron_density', 'delta_E_max']}
        dominant_element = max(element_percentages, key=element_percentages.get) if element_percentages else 'none'
        strata_keys.append(dominant_element)

    # --- LOGIKA PEMBAGIAN MANUAL YANG SEPENUHNYA BARU ---

    print(f"[{get_timestamp()}] Mengelompokkan sampel berdasarkan kelas...")
    strata_groups = defaultdict(list)
    for i, key in enumerate(strata_keys):
        strata_groups[key].append(i)

    # Siapkan list untuk menampung indeks final
    final_train_idx, final_val_idx, final_test_idx = [], [], []
    
    # Persentase split
    train_ratio = 1.0 - CONFIG["split"]["test_size"] # e.g., 0.7
    val_ratio = CONFIG["split"]["test_size"] * CONFIG["split"]["validation_size"] # e.g., 0.3 * 0.5 = 0.15
    # test_ratio dihitung dari sisa

    print(f"[{get_timestamp()}] Melakukan pembagian manual untuk setiap kelas...")
    for key, indices in tqdm(strata_groups.items(), desc="Splitting per class"):
        n_samples_in_class = len(indices)
        np.random.shuffle(indices) # Acak urutan sampel di dalam kelas

        # ATURAN KUNCI: Jika kelas terlalu kecil, masukkan semua ke training set
        if n_samples_in_class < 3:
            final_train_idx.extend(indices)
            continue

        # Hitung jumlah sampel untuk setiap set
        n_train = math.ceil(n_samples_in_class * train_ratio)
        n_val = math.floor(n_samples_in_class * val_ratio)
        
        # Alokasikan indeks berdasarkan perhitungan
        class_train_idx = indices[:n_train]
        class_val_idx = indices[n_train : n_train + n_val]
        class_test_idx = indices[n_train + n_val:] # Sisanya untuk test

        # Pastikan test set tidak kosong jika val mengambil sisa pembulatan
        if len(class_test_idx) == 0 and len(class_val_idx) > 0:
            # Pindahkan satu sampel dari val ke test
            class_test_idx.append(class_val_idx.pop())

        # Tambahkan indeks kelas ini ke list final
        final_train_idx.extend(class_train_idx)
        final_val_idx.extend(class_val_idx)
        final_test_idx.extend(class_test_idx)

    # Acak urutan final dari setiap set
    np.random.shuffle(final_train_idx)
    np.random.shuffle(final_val_idx)
    np.random.shuffle(final_test_idx)

    print(f"[{get_timestamp()}] Pembagian manual selesai: Train={len(final_train_idx)}, Validation={len(final_val_idx)}, Test={len(final_test_idx)}")
    
    # 7. Tulis file HDF5 baru (Sama)
    print(f"[{get_timestamp()}] Menyimpan dataset baru ke: {new_path}...")
    with h5py.File(new_path, 'w') as f:
        f.create_dataset('wavelengths', data=wavelengths)
        
        def write_group(group_name, indices):
            # Konversi list ke numpy array untuk advanced indexing
            indices_np = np.array(indices, dtype=int)
            grp = f.create_group(group_name)
            grp.create_dataset('spectra', data=all_spectra[indices_np], compression='gzip')
            grp.create_dataset('labels', data=all_labels[indices_np], compression='gzip')
            grp.create_dataset('atom_percentages', data=all_atom_percentages_str[indices_np], compression='gzip')

        write_group('train', final_train_idx)
        write_group('validation', final_val_idx)
        write_group('test', final_test_idx)
        
        f.attrs['description'] = 'Dataset with MANUAL robust stratified split.'
        f.attrs['creation_date'] = get_timestamp()

    print(f"[{get_timestamp()}] Pembuatan dataset stratified selesai.")

# --- FUNGSI VERIFIKASI (dari kode Anda, sedikit disesuaikan) ---
def compute_class_distribution(labels, element_map, split_name):
    """Hitung distribusi kelas berdasarkan label one-hot dan cetak laporan."""
    num_samples, num_timesteps, num_classes = labels.shape
    class_counts = np.sum(labels, axis=(0, 1))
    total_positions = num_samples * num_timesteps
    
    print(f"\nLaporan Distribusi Dataset - {split_name}")
    print(f"  Jumlah Sampel: {num_samples:,}")
    print(f"  Total Posisi (Sampel x Timesteps): {total_positions:,}")
    
    dist_report = {}
    for class_name, one_hot in element_map.items():
        class_idx = np.argmax(one_hot)
        count = int(class_counts[class_idx])
        percentage = (count / total_positions) * 100 if total_positions > 0 else 0
        dist_report[class_name] = (count, percentage)
        
    # Urutkan berdasarkan persentase
    sorted_dist = sorted(dist_report.items(), key=lambda item: item[1][0], reverse=True)
    
    print("\n  Distribusi Kelas (Posisi Berlabel):")
    for class_name, (count, percentage) in sorted_dist:
        print(f"    {class_name:<15} : {count:12,d} posisi ({percentage:6.2f}%)")

def verify_stratified_distribution(stratified_path, element_map_path):
    """Memuat dataset yang sudah distratifikasi dan memeriksa distribusinya."""
    print(f"[{get_timestamp()}] Memulai verifikasi distribusi dari {stratified_path}...")
    
    try:
        with open(element_map_path, 'r') as f:
            element_map = json.load(f)
    except Exception as e:
        print(f"[{get_timestamp()}] Gagal memuat element_map: {e}")
        return

    try:
        with h5py.File(stratified_path, 'r') as f:
            train_labels = f['train/labels'][:]
            val_labels = f['validation/labels'][:]
            test_labels = f['test/labels'][:]
            print(f"[{get_timestamp()}] Dataset dimuat: train={train_labels.shape}, validation={val_labels.shape}, test={test_labels.shape}")

            compute_class_distribution(train_labels, element_map, "TRAIN")
            compute_class_distribution(val_labels, element_map, "VALIDATION")
            compute_class_distribution(test_labels, element_map, "TEST")

            total_samples = train_labels.shape[0] + val_labels.shape[0] + test_labels.shape[0]
            print(f"\n[{get_timestamp()}] Verifikasi Selesai. Total Sampel: {total_samples:,}")
    except Exception as e:
        print(f"[{get_timestamp()}] Gagal memuat atau memproses dataset yang distratifikasi: {e}")


# --- FUNGSI UTAMA ---
if __name__ == "__main__":
    # Langkah 1: Buat dataset yang distratifikasi
    try:
        create_stratified_dataset(
            CONFIG["data"]["original_dataset_path"],
            CONFIG["data"]["stratified_dataset_path"],
            CONFIG["data"]["element_map_path"]
        )
    except Exception as e:
        print(f"\n[{get_timestamp()}] Terjadi kesalahan saat membuat dataset: {e}")
    
    print("\n" + "="*80 + "\n")
    
    # Langkah 2: Verifikasi distribusi dataset yang baru dibuat
    try:
        verify_stratified_distribution(
            CONFIG["data"]["stratified_dataset_path"],
            CONFIG["data"]["element_map_path"]
        )
    except Exception as e:
        print(f"\n[{get_timestamp()}] Terjadi kesalahan saat memverifikasi dataset: {e}")

[2025-06-14 01:25:49] Memulai pembuatan dataset stratified dengan metode manual...
[2025-06-14 01:25:49] Memuat data...
[2025-06-14 01:26:09] Membuat kunci stratifikasi...


Parsing Metadata: 100%|██████████| 10000/10000 [00:00<00:00, 132280.30it/s]


[2025-06-14 01:26:09] Mengelompokkan sampel berdasarkan kelas...
[2025-06-14 01:26:09] Melakukan pembagian manual untuk setiap kelas...


Splitting per class: 100%|██████████| 24/24 [00:00<00:00, 7377.84it/s]

[2025-06-14 01:26:09] Pembagian manual selesai: Train=7012, Validation=1487, Test=1501
[2025-06-14 01:26:09] Menyimpan dataset baru ke: /Volumes/Data/birrulwldain/spectral_dataset-35_s.h5...


[2025-06-14 01:26:51] Pembuatan dataset stratified selesai.


[2025-06-14 01:26:51] Memulai verifikasi distribusi dari /Volumes/Data/birrulwldain/spectral_dataset-35_s.h5...
[2025-06-14 01:27:08] Dataset dimuat: train=(7012, 4096, 35), validation=(1487, 4096, 35), test=(1501, 4096, 35)

Laporan Distribusi Dataset - TRAIN
  Jumlah Sampel: 7,012
  Total Posisi (Sampel x Timesteps): 28,721,152

  Distribusi Kelas (Posisi Berlabel):
    background      :   14,285,300 posisi ( 49.74%)
    Fe_2            :    2,862,803 posisi (  9.97%)
    Ar_2            :    1,703,051 posisi (  5.93%)
    Ti_2            :    1,671,364 posisi (  5.82%)
    Co_2            :    1,255,059 posisi (  4.37%)
    Mn_2            :      715,063 posisi (  2.49%)
    Al_2            :      581,246 posisi (  2.02%)
    Na_2            :      581,155 posisi (  2.02%)
    C_2             :      477,040 posisi (  1.66%)
    N_1             :      474,784 posisi (  1.65%)
    Si_2            :      462,936 posisi (  1.